In [0]:
from pyspark.sql import functions as F

# Base profile: all members with unredeemed tickets as of July 15, 2026
# recognitions_left < 100 mirrors the Ticket Liability Analysis test-account exclusion
# (max legitimate balance = 59; values >= 100 are test/dummy accounts)
T_profile = (
    spark.table("cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot")
    .filter(
        (F.col("snapshot_date_id") == "20260715") &
        (F.col("recognitions_left") > 0) &
        (F.col("recognitions_left") < 100)
    )
)

# Lightweight version for downstream credit calculations
df_T1 = T_profile.select("cde_id", "renewal_status", "plan_id", "recognitions_left", "annual_current_term_end_datetime")



## CineClub Ticket Redemption Rate Analysis — Pre-Price Change Impact

### Purpose
Quantify the outstanding CineClub free ticket inventory and model how quickly tickets will exit the market at current redemption behavior. This supports 2027 budget planning by estimating the revenue exposure from pre-price-change ($9.99) tickets still in circulation.

### Data Sources
| Table | Role |
|-------|------|
| `cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot` | Member profiles, plan status, tickets remaining |
| `cpx_dataanalytics_gold.customer360.c360_f_transaction` | Transaction history (visits, ticket redemptions) |

### Key Definitions
- **Tickets in Account** — `recognitions_left`: credited but not yet redeemed free tickets
- **$9.99 Tickets to be Credited** — future monthly credits for annual/gift-plan members until term end (post Jul 15, 2026)
- **Free Ticket Redemption (Type 6)** — CineClub free ticket used at box office (draws from balance)
- **Member Priced Ticket (Type 5)** — companion ticket purchased at discounted member price (does NOT draw from balance)
- **Visit** — unique combination of CDE_ID + VisitDateId + LocationId
- **CC Visit** — visit where `CineClubFLAG = 'Yes'`

### Analysis Steps
1. **Credit Forecast** — Month-by-month schedule of new $9.99 tickets to be credited (annual/gift-plan only)
2. **Redemption Behavior** — Per-member Free Ticket redemption rate and Member Priced attach rate (2021–2026)
3. **Clearance Simulation** — Member-level velocity model projecting when tickets leave the market
4. **Companion Attach Rate** — Historical and projected Member Priced ticket attachment rate for credit-eligible members

### Insight: Credit Schedule (Pre-Price-Change Tickets)

This shows the month-by-month inflow of **new $9.99 tickets** being credited to annual and gift-plan members.

Member pool aligns with the **Ticket Liability Analysis** methodology:
- **Effective renewal date** is used throughout — gift-plan `Active_Non_Renewing` members with stale `renewal_datetime` are corrected via `annual_current_term_end_datetime`, as are `annual1` mid-term stackers where the gift pushed the end date 12+ months beyond `renewal_datetime`
- **Monthly-to-annual upgrade cohort** (`Active_Annual_Upgrade` on `monthly1`) is included — their annual plan starts at `renewal_datetime`, so effective end = `renewal_datetime + 12 months`

- **Pending (unstacked) gift cards** are now included: members with gift cards redeemed outside Chargebee's 12-month auto-stack window have their `end_date` extended by +12 months per pending gift before the credit schedule is computed. This aligns with the **Contingent Liability** section of the Ticket Liability Analysis. Members whose `end_date` was already corrected via `annual_current_term_end_datetime` are excluded from this scan to avoid double-counting.

**Interpretation:** The inflow of new pre-price-change tickets is a finite, declining stream. The bulk lands within 6 months, meaning the market exposure from new credits is time-limited. The pending gift cohort adds a small tail of credits that extend slightly beyond the base cohort's term horizon.

In [0]:
# Goal: show how many tickets are CREDITED in each calendar month from July 15 2026.
# Uses effective_renewal_datetime from the Ticket Liability Analysis methodology:
#   - Gift-plan ANR with stale renewal_datetime → corrected via annual_current_term_end_datetime
#   - Annual1 mid-term stackers (gap >= 12m)    → corrected via annual_current_term_end_datetime
#   - Monthly-to-annual upgrade cohort          → effective end = renewal_datetime + 12 months
#   - Pending (unstacked) gift cards            → end_date extended +12m per pending gift
#     (aligned with Ticket Liability Analysis Section 3: Contingent Liability)
#
# Approach:
#   1. Per member: compute effective end date (incl. pending gift extension), first credit date, credit count
#   2. Explode into one row per credit event using add_months(first_credit, offset)
#   3. Aggregate by the calendar month the credit falls in

SNAPSHOT_DATE = "2026-07-15"
start_date = F.to_date(F.lit(SNAPSHOT_DATE))

# --- Annual and gift-plan members with effective renewal date correction ---
# was_stacking_corrected: True when end_date was derived from annual_current_term_end_datetime.
# These members are excluded from the pending-gift scan below to avoid double-counting
# (their prior stacking correction was validated against subscription_changed events).
df_annual_gift = (
    spark.table("cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot")
    .filter(
        (F.col("snapshot_date_id") == "20260715") &
        F.col("renewal_status").like("Active%") &
        (
            F.col("plan_id").ilike("%gift%") |
            F.col("plan_id").ilike("%annual%")
        )
    )
    .select("cde_id", "cc_user_profile_id", "renewal_status", "plan_id",
            "renewal_datetime", "annual_current_term_end_datetime")
    .withColumn(
        "was_stacking_corrected",
        F.when(
            (F.col("renewal_status") == "Active_Non_Renewing") &
            F.col("plan_id").ilike("%gift%") &
            (F.col("renewal_datetime") < F.lit(SNAPSHOT_DATE).cast("timestamp")) &
            (F.col("annual_current_term_end_datetime") > F.col("renewal_datetime")),
            F.lit(True)
        ).when(
            F.col("plan_id").ilike("%annual%") &
            (F.col("annual_current_term_end_datetime") > F.add_months(F.col("renewal_datetime"), 11)),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .withColumn(
        "end_date",
        F.when(
            # Gift-plan ANR: renewal_datetime is stale (past); annual_current_term_end corrects it
            (F.col("renewal_status") == "Active_Non_Renewing") &
            F.col("plan_id").ilike("%gift%") &
            (F.col("renewal_datetime") < F.lit(SNAPSHOT_DATE).cast("timestamp")) &
            (F.col("annual_current_term_end_datetime") > F.col("renewal_datetime")),
            F.col("annual_current_term_end_datetime").cast("date")
        ).when(
            # Annual1 mid-term stackers: gift pushed end date 12+ months beyond renewal_datetime
            F.col("plan_id").ilike("%annual%") &
            (F.col("annual_current_term_end_datetime") > F.add_months(F.col("renewal_datetime"), 11)),
            F.col("annual_current_term_end_datetime").cast("date")
        ).otherwise(F.col("renewal_datetime").cast("date"))
    )
)

# --- Monthly-to-annual upgrade cohort (annual plan starts at next renewal_datetime) ---
df_upgrades = (
    spark.table("cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot")
    .filter(
        (F.col("snapshot_date_id") == "20260715") &
        (F.col("renewal_status") == "Active_Annual_Upgrade") &
        (F.col("plan_id") == "monthly1")
    )
    .select("cde_id", "cc_user_profile_id", "renewal_status", "plan_id", "renewal_datetime")
    .withColumn("was_stacking_corrected", F.lit(False))
    .withColumn("end_date", F.add_months(F.col("renewal_datetime"), 12).cast("date"))
)

# --- Combine cohorts ---
df_combined = (
    df_annual_gift.select("cde_id", "cc_user_profile_id", "was_stacking_corrected", "renewal_status", "plan_id", "end_date")
    .unionByName(df_upgrades.select("cde_id", "cc_user_profile_id", "was_stacking_corrected", "renewal_status", "plan_id", "end_date"))
    .filter(F.col("end_date") >= start_date)
)

# --- Pending (unstacked) gift card extension ---
# Gifts redeemed outside Chargebee's 12-month auto-stack window are not yet applied.
# FUTURE:   redemption_date >= end_date  (gift arrived after current term closes)
# ORPHANED: redemption_date < add_months(end_date, -12)  (gift falls outside auto-stack window)
# Only scan members whose end_date was NOT already corrected from annual_current_term_end_datetime
# to prevent double-counting confirmed-applied gifts.
df_gifting_raw = (
    spark.table("cpx_dataanalytics_silver.cineclub.gifting")
    .filter(
        (F.col("is_redeemed") == True) &
        (F.col("redeemer_plan_id").isin(["gift-plan", "annual1"]))
    )
    .select(
        F.col("redeemer_id").alias("cc_user_profile_id_g"),
        F.to_date(F.col("redemption_date")).alias("redemption_date")
    )
)

df_pending_counts = (
    df_combined
    .filter(F.col("was_stacking_corrected") == False)
    .join(df_gifting_raw, F.col("cc_user_profile_id") == F.col("cc_user_profile_id_g"), "inner")
    .drop("cc_user_profile_id_g")
    .filter(
        (F.col("redemption_date") >= F.col("end_date")) |
        (F.col("redemption_date") < F.add_months(F.col("end_date"), -12))
    )
    .groupBy("cde_id")
    .agg(F.count("*").alias("pending_gifts_count"))
)

pending_member_count = df_pending_counts.count()
pending_total_gifts  = df_pending_counts.agg(F.sum("pending_gifts_count")).collect()[0][0] or 0
print(f"Members with pending (unstacked) gifts   : {pending_member_count:,}")
print(f"Contingent ticket exposure (once applied): ~{int(pending_total_gifts) * 12:,} tickets")

# Extend end_date for pending-gift members; drop join scaffolding
df_combined_ext = (
    df_combined
    .join(df_pending_counts, "cde_id", "left")
    .fillna({"pending_gifts_count": 0})
    .withColumn(
        "end_date",
        F.add_months(F.col("end_date"), F.col("pending_gifts_count").cast("int") * 12)
    )
    .drop("was_stacking_corrected", "pending_gifts_count", "cc_user_profile_id")
)

# --- Compute per-member credit schedule ---
df_members = (
    df_combined_ext
    .withColumn("renewal_day", F.dayofmonth(F.col("end_date")))
    .withColumn(
        "999_Tickets_Credited",
        F.greatest(
            F.lit(0),
            F.ceil(F.months_between(F.col("end_date"), start_date) - 1).cast("long")
        )
    )
    .filter(F.col("999_Tickets_Credited") > 0)
    .withColumn("july_renewal", F.make_date(F.lit(2026), F.lit(7), F.col("renewal_day")))
    .withColumn(
        "first_credit",
        F.when(F.col("july_renewal") >= start_date, F.col("july_renewal"))
         .otherwise(F.add_months(F.col("july_renewal"), 1))
    )
    .drop("july_renewal")
)

# Step 2 — explode: one row per monthly credit event per member
df_credits = (
    df_members
    .withColumn(
        "month_offset",
        F.explode(F.sequence(F.lit(0), (F.col("999_Tickets_Credited") - 1).cast("int")))
    )
    .withColumn("credit_date", F.add_months(F.col("first_credit"), F.col("month_offset")))
    .withColumn("credit_month_sort", F.date_format(F.col("credit_date"), "yyyyMM").cast("int"))
    .withColumn("credit_month",      F.date_format(F.col("credit_date"), "MMMM yyyy"))
)

# Step 3 — aggregate: tickets credited per calendar month
df_tickets_by_credit_month = (
    df_credits
    .groupBy("credit_month", "credit_month_sort")
    .agg(F.count("cde_id").alias("Tickets_Credited"))
    .orderBy("credit_month_sort")
    .drop("credit_month_sort")
)


In [0]:
display(df_tickets_by_credit_month)

In [0]:
df_tickets_per_member = (
    df_members
    .select("cde_id", "plan_id", "renewal_status", "end_date", "renewal_day", "first_credit", "999_Tickets_Credited")
    .orderBy(F.col("999_Tickets_Credited").desc())
)

display(df_tickets_per_member)

### Insight: Per-Member Redemption Rates

**Free Ticket Redemption Rate** = Tickets Redeemed / (Redeemed + Still in Account)
- Measures what % of all tickets ever credited have actually been used
- High rates (>0.8) indicate members who redeem almost every credit they receive
- Low rates (<0.5) indicate ticket hoarders or lapsed members

**Member Priced Attach Rate** = Companion Tickets Purchased / CC Visits
- Measures how often members bring a companion when using CineClub benefits
- Rate > 1.0 means they buy multiple companions per visit
- Rate = 0 means they never buy companion tickets

**Note:** These rates are based on 2025 behavior (full year). Members with no 2025 activity show velocity = 0 in subsequent simulations. This is conservative — some may resume activity but we do not assume so.

In [0]:
# Step 4: Per-member redemption behavior from 2021 onwards
years = [2021, 2022, 2023, 2024, 2025, 2026]

# --- Base members: ALL exposure — tickets currently in account OR future credits still to arrive.
#
# Fix aligned with Ticket Liability Analysis: the original T_profile filter
# (recognitions_left > 0) excluded credit-eligible members who had redeemed their
# current balance to zero but still have months of credits remaining. Those members
# have real forward exposure and must be included in the velocity simulation.
#
# Group A: df_tickets_per_member (all annual/gift/upgrade members with future credits)
#          → LEFT JOIN T_profile for current account balance (0 if fully redeemed)
# Group B: T_profile members NOT in the credit schedule (monthly/lapsed members holding
#          a residual ticket balance with no further credits expected)
df_base = (
    # Group A
    df_tickets_per_member
    .select(
        "cde_id", "plan_id", "renewal_status",
        F.col("999_Tickets_Credited").alias("Tickets_to_be_Credited")
    )
    .join(
        T_profile.select(
            F.col("cde_id").alias("cde_id_tp"),
            F.col("recognitions_left").alias("Tickets_in_Account")
        ),
        F.col("cde_id") == F.col("cde_id_tp"),
        "left"
    )
    .drop("cde_id_tp")
    .fillna({"Tickets_in_Account": 0})
    # Group B: residual-balance holders outside the forward credit schedule
    .unionByName(
        T_profile
        .select(
            "cde_id", "plan_id", "renewal_status",
            F.col("recognitions_left").alias("Tickets_in_Account")
        )
        .join(
            df_tickets_per_member.select(F.col("cde_id").alias("cde_id_anti")),
            F.col("cde_id") == F.col("cde_id_anti"),
            "left_anti"
        )
        .withColumn("Tickets_to_be_Credited", F.lit(0))
    )
)

# --- Transaction data from 2021 onwards, cast CDE_ID to string for join
df_txn = (
    spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")
    .filter(F.col("VisitDateId") >= 20210101)
    .withColumn("Year", F.substring(F.col("VisitDateId").cast("string"), 1, 4).cast("int"))
    .withColumn("cde_id_str", F.col("CDE_ID").cast("string"))
)

# --- All Visits per member per year (unique CDE_ID + VisitDateId + LocationId)
df_all_visits = (
    df_txn
    .select("cde_id_str", "VisitDateId", "LocationId", "Year")
    .dropDuplicates(["cde_id_str", "VisitDateId", "LocationId"])
    .groupBy("cde_id_str", "Year")
    .agg(F.count("*").alias("visits"))
    .groupBy("cde_id_str")
    .pivot("Year", years)
    .agg(F.first("visits"))
)
for yr in years:
    df_all_visits = df_all_visits.withColumnRenamed(str(yr), f"All_Visits_{yr}")

# --- CC Visits per member per year (CineClubFLAG = 'Yes', unique visit)
df_cc_visits = (
    df_txn
    .filter(F.col("CineClubFLAG") == "Yes")
    .select("cde_id_str", "VisitDateId", "LocationId", "Year")
    .dropDuplicates(["cde_id_str", "VisitDateId", "LocationId"])
    .groupBy("cde_id_str", "Year")
    .agg(F.count("*").alias("cc_visits"))
    .groupBy("cde_id_str")
    .pivot("Year", years)
    .agg(F.first("cc_visits"))
)
for yr in years:
    df_cc_visits = df_cc_visits.withColumnRenamed(str(yr), f"CC_Visits_{yr}")

# --- Free Tickets Redeemed (CineClub_Discount_Type_ID = 6) - SUM(Quantity)
df_free_tickets = (
    df_txn
    .filter(F.col("CineClub_Discount_Type_ID") == 6)
    .groupBy("cde_id_str", "Year")
    .agg(F.sum("Quantity").alias("free_tix"))
    .groupBy("cde_id_str")
    .pivot("Year", years)
    .agg(F.first("free_tix"))
)
for yr in years:
    df_free_tickets = df_free_tickets.withColumnRenamed(str(yr), f"Free_Tickets_{yr}")

# --- Member Priced Tickets (CineClub_Discount_Type_ID = 5) - SUM(Quantity)
df_member_priced = (
    df_txn
    .filter(F.col("CineClub_Discount_Type_ID") == 5)
    .groupBy("cde_id_str", "Year")
    .agg(F.sum("Quantity").alias("member_tix"))
    .groupBy("cde_id_str")
    .pivot("Year", years)
    .agg(F.first("member_tix"))
)
for yr in years:
    df_member_priced = df_member_priced.withColumnRenamed(str(yr), f"Member_Priced_{yr}")

# --- P12M Free Tickets redeemed (Jul 15 2025 – Jul 14 2026)
# Exact 12-month window ending the day before the snapshot — no partial-month normalisation needed.
# Used as the velocity basis in place of Full-Year 2025 ÷ 12: captures any behavioural
# shift around the price-change announcement and reflects the most recent redemption patterns.
df_p12m_free = (
    df_txn
    .filter((F.col("VisitDateId") >= 20250715) & (F.col("VisitDateId") <= 20260714))
    .filter(F.col("CineClub_Discount_Type_ID") == 6)
    .groupBy("cde_id_str")
    .agg(F.sum("Quantity").alias("P12M_Free_Tickets"))
)

# --- P12M CC Visits (Jul 15 2025 – Jul 14 2026, unique visit)
df_p12m_cc = (
    df_txn
    .filter((F.col("VisitDateId") >= 20250715) & (F.col("VisitDateId") <= 20260714))
    .filter(F.col("CineClubFLAG") == "Yes")
    .select("cde_id_str", "VisitDateId", "LocationId")
    .dropDuplicates(["cde_id_str", "VisitDateId", "LocationId"])
    .groupBy("cde_id_str")
    .agg(F.count("*").alias("P12M_CC_Visits"))
)

# --- P12M Member Priced Tickets (Jul 15 2025 – Jul 14 2026)
# Paired with P12M_CC_Visits so the attach rate uses the same window as the visit velocity.
df_p12m_member_priced = (
    df_txn
    .filter((F.col("VisitDateId") >= 20250715) & (F.col("VisitDateId") <= 20260714))
    .filter(F.col("CineClub_Discount_Type_ID") == 5)
    .groupBy("cde_id_str")
    .agg(F.sum("Quantity").alias("P12M_Member_Priced"))
)

# --- Final join
df_step4 = (
    df_base
    .join(df_all_visits,   df_base.cde_id == df_all_visits.cde_id_str,   "left").drop(df_all_visits.cde_id_str)
    .join(df_cc_visits,    df_base.cde_id == df_cc_visits.cde_id_str,    "left").drop(df_cc_visits.cde_id_str)
    .join(df_free_tickets, df_base.cde_id == df_free_tickets.cde_id_str, "left").drop(df_free_tickets.cde_id_str)
    .join(df_member_priced,df_base.cde_id == df_member_priced.cde_id_str,"left").drop(df_member_priced.cde_id_str)
    .join(df_p12m_free,    df_base.cde_id == df_p12m_free.cde_id_str,    "left").drop(df_p12m_free.cde_id_str)
    .join(df_p12m_cc,           df_base.cde_id == df_p12m_cc.cde_id_str,           "left").drop(df_p12m_cc.cde_id_str)
    .join(df_p12m_member_priced, df_base.cde_id == df_p12m_member_priced.cde_id_str, "left").drop(df_p12m_member_priced.cde_id_str)
)



In [0]:
# --- Compute redemption rates from existing columns
years = [2021, 2022, 2023, 2024, 2025, 2026]

# Total Free Tickets Redeemed (sum across years)
free_ticket_cols = [F.coalesce(F.col(f"Free_Tickets_{yr}"), F.lit(0)) for yr in years]
df_step4 = df_step4.withColumn("Total_Free_Tickets_Redeemed", sum(free_ticket_cols))

# Free Ticket Redemption Rate = Redeemed / (Redeemed + Tickets_in_Account)
df_step4 = df_step4.withColumn(
    "Free_Ticket_Redemption_Rate",
    F.when(
        (F.col("Total_Free_Tickets_Redeemed") + F.col("Tickets_in_Account")) > 0,
        F.round(
            F.col("Total_Free_Tickets_Redeemed") / 
            (F.col("Total_Free_Tickets_Redeemed") + F.col("Tickets_in_Account")),
            4
        )
    )
)

# Total Member Priced Tickets (sum across years)
member_priced_cols = [F.coalesce(F.col(f"Member_Priced_{yr}"), F.lit(0)) for yr in years]
df_step4 = df_step4.withColumn("Total_Member_Priced", sum(member_priced_cols))

# Total CC Visits (sum across years)
cc_visit_cols = [F.coalesce(F.col(f"CC_Visits_{yr}"), F.lit(0)) for yr in years]
df_step4 = df_step4.withColumn("Total_CC_Visits", sum(cc_visit_cols))

# Member Priced Attach Rate = Total Member Priced / Total CC Visits
df_step4 = df_step4.withColumn(
    "Member_Priced_Attach_Rate",
    F.when(
        F.col("Total_CC_Visits") > 0,
        F.round(
            F.col("Total_Member_Priced") / F.col("Total_CC_Visits"),
            4
        )
    )
)

display(df_step4.select(
    "cde_id", "plan_id", "renewal_status",
    "Tickets_in_Account", "Tickets_to_be_Credited",
    "Total_Free_Tickets_Redeemed", "Free_Ticket_Redemption_Rate",
    "Total_Member_Priced", "Total_CC_Visits", "Member_Priced_Attach_Rate"
))

In [0]:
# --- Time to Clear: Member-level velocity simulation
import pandas as pd
import numpy as np

# 1. Per-member: monthly velocity based on P12M redemptions (Jul 15 2025 – Jul 14 2026)
# P12M is a complete 12-month window so dividing by 12 gives an unbiased monthly rate.
df_member_clearance = (
    df_step4
    .withColumn("Monthly_Velocity", F.coalesce(F.col("P12M_Free_Tickets"), F.lit(0)) / 12)
    .withColumn("Total_Exposure", F.col("Tickets_in_Account") + F.col("Tickets_to_be_Credited"))
    .select("cde_id", "plan_id", "Tickets_in_Account", "Tickets_to_be_Credited", 
            "Total_Exposure", "Monthly_Velocity")
)

# Collect to pandas for simulation (~274K members)
members = df_member_clearance.toPandas()
members["Monthly_Velocity"] = members["Monthly_Velocity"].astype(float)
members["Total_Exposure"] = members["Total_Exposure"].astype(float)
members["Tickets_in_Account"] = members["Tickets_in_Account"].astype(float)
members["Tickets_to_be_Credited"] = members["Tickets_to_be_Credited"].astype(float)

# Get the credit schedule per member (months of credits remaining)
# Credits come 1/month for Tickets_to_be_Credited months
members["Credit_Months"] = members["Tickets_to_be_Credited"].astype(int)

# 2. Summary stats
total_exposure = members["Total_Exposure"].sum()
total_in_account = members["Tickets_in_Account"].sum()
total_to_credit = members["Tickets_to_be_Credited"].sum()
active_redeemers = members[members["Monthly_Velocity"] > 0]
non_redeemers = members[members["Monthly_Velocity"] == 0]

print("=" * 70)
print("MEMBER-LEVEL VELOCITY ANALYSIS")
print("=" * 70)
print(f"\nTotal Members:              {len(members):,}")
print(f"Active Redeemers (vel > 0): {len(active_redeemers):,}")
print(f"Non-Redeemers (vel = 0):    {len(non_redeemers):,}")
print(f"\nTotal Tickets in Account:   {total_in_account:,.0f}")
print(f"Total Tickets to Credit:    {total_to_credit:,.0f}")
print(f"Total Exposure:             {total_exposure:,.0f}")
print(f"\nTickets held by non-redeemers: {non_redeemers['Total_Exposure'].sum():,.0f} (WILL NEVER CLEAR at current rate)")
print(f"Tickets held by active redeemers: {active_redeemers['Total_Exposure'].sum():,.0f}")
print(f"\nAggregate Monthly Velocity: {active_redeemers['Monthly_Velocity'].sum():,.0f} tickets/month")

# 3. Month-by-month simulation using individual member velocity
# For each month: each member redeems min(velocity, remaining_balance)
# and receives 1 credit if they still have credits to receive

max_months = 60  # simulate up to 5 years
balance = members["Tickets_in_Account"].values.copy()
velocity = members["Monthly_Velocity"].values.copy()
credits_remaining = members["Credit_Months"].values.copy()

monthly_results = []

for t in range(1, max_months + 1):
    # Credits arrive for members who still have credits to receive
    credits_this_month = (credits_remaining > 0).astype(float)
    balance += credits_this_month
    credits_remaining = np.maximum(credits_remaining - 1, 0)
    
    # Redemptions: min(velocity, balance)
    redeemed = np.minimum(velocity, balance)
    balance = np.maximum(balance - redeemed, 0)
    
    total_balance = balance.sum()
    total_redeemed = redeemed.sum()
    total_credits = credits_this_month.sum()
    
    monthly_results.append({
        "Month": t,
        "Calendar_Month": pd.Timestamp("2026-07-01") + pd.DateOffset(months=t-1),
        "Credits_In": total_credits,
        "Redeemed": total_redeemed,
        "Remaining_Balance": total_balance
    })
    
    if total_balance <= 0:
        break

df_sim = pd.DataFrame(monthly_results)
df_sim["Calendar_Month"] = df_sim["Calendar_Month"].dt.strftime("%B %Y")

# 4. Print results
print(f"\n\n{'='*70}")
print("MONTH-BY-MONTH CLEARANCE SIMULATION")
print(f"{'='*70}")
print(f"\n{'Month':<5} {'Calendar':<18} {'Credits In':>12} {'Redeemed':>12} {'Remaining':>14}")
print("-" * 65)

for _, row in df_sim.iterrows():
    print(f"{int(row['Month']):<5} {row['Calendar_Month']:<18} {row['Credits_In']:>12,.0f} {row['Redeemed']:>12,.0f} {row['Remaining_Balance']:>14,.0f}")

if df_sim["Remaining_Balance"].iloc[-1] > 0:
    print(f"\n** After {max_months} months, {df_sim['Remaining_Balance'].iloc[-1]:,.0f} tickets still remain (held by non/low-redeemers) **")
else:
    clear_month = df_sim[df_sim["Remaining_Balance"] <= 0].iloc[0]
    print(f"\n** Market fully cleared in month {int(clear_month['Month'])} ({clear_month['Calendar_Month']}) **")

### Insight: Ticket Clearance Simulation

This simulation uses **individual member velocity** (P12M monthly redemption rate: Jul 15 2025 – Jul 14 2026) to project when tickets exit the market.

**Population refinement (aligned with Ticket Liability Analysis):** The base population now includes credit-eligible members with `recognitions_left = 0` at snapshot date — members who had fully redeemed their current balance but still have months of forward credits remaining. The prior version used `T_profile` (filtered to `recognitions_left > 0`) as the base, which omitted these members entirely. Their velocity may be 0 (conservative), but their `Tickets_to_be_Credited` is real forward exposure. Numbers below will update on re-run.

**Key findings (P12M velocity run — 284,981 members):**
- **Total exposure: 1,928,331 tickets** — 1,171,803 currently in accounts + 756,528 still to be credited
- **Active Redeemers: 244,705** vs 40,276 non-redeemers — P12M captures members who redeemed in early 2026 but had zero 2025 activity, reducing apparent dormancy vs the prior full-year-2025 basis
- **Month 1 net: ~155K redeemed, ~101K credited** — stock begins declining immediately at 157K tickets/month aggregate velocity
- **Deceleration is steep:** new credits cease after June 2030; monthly redemptions drop from ~155K → ~14K by month 24
- **40,276 non-redeemers** (vel = 0 in P12M) hold **278,939 tickets** that will never clear at current behavior
- After 60 months, **~425,240 tickets still remain** (~22% of original exposure)
- **Contingent liability (pending gifts):** 3,998 members with ~57,516 tickets once gifts are applied — already baked into the 1,928,331 figure above
- **Caveat:** the P12M window overlaps with any pre-announcement acceleration (Jan–Jul 2026); velocity may be slightly overstated if members front-loaded redemptions ahead of the price change

**Interpretation:** The market will NOT fully clear organically, but P12M velocity paints a materially less pessimistic picture than the full-year-2025 basis — residual drops from ~32% to ~22% of exposure. The remaining ~425K tickets are structurally held in genuinely dormant accounts. Any budget model should treat ~280K–425K as the permanent residual range.

### Insight: Historical Member Priced Attach Rate

This shows the actual companion ticket purchase rate per CC visit, month-over-month, for the credit-eligible cohort (annual/gift-plan members only).

**Trends:**
- The rate has **grown steadily** from ~0.45 (2022) to ~0.60–0.65 (2026), suggesting increasing companion-ticket engagement
- **Seasonality is visible:** summer months (Jun–Jul) and holiday periods (Nov–Dec) show peaks, while January dips
- The denominator adjusts for plan expiry — only members with `end_date >= month` are counted

**Interpretation:** Credit-eligible members are bringing companions more frequently over time. This upward trend means the companion ticket revenue associated with pre-price-change members is likely to remain strong during their remaining term. Ignore Aug 2026+ data (tiny sample sizes from near-expired plans).

### Insight: Projected Companion Attach Rate (Forward-Looking)

This projects the expected Member Priced ticket purchases month-by-month as the credit-eligible cohort shrinks.

**Key findings (P12M CC visit velocity + P12M attach rate — Jul 15 2025 – Jul 14 2026):**
- The attach rate holds **stable at ~0.65** throughout the peak period (Jul 2026 – Jun 2027), stepping down to ~0.59–0.60 after Jul 2027 as the higher-engagement cohort renews out
- **Total expected companion tickets: ~507K** over the full credit period
- **Total expected CC visits: ~779K** from these members
- **Overall weighted attach rate: 0.651**

**Assumption:** Both the CC visit velocity (`P12M_CC_Visits ÷ 12`) and the per-member attach rate (`P12M_Member_Priced / P12M_CC_Visits`) are now drawn from the same trailing 12-month window, ensuring internal consistency. Members with zero P12M CC visits get an attach rate of 0.

**Budget implication:** ~779K expected CC visits from credit-eligible members during their remaining term, generating ~507K companion ticket purchases at member price.

In [0]:
# Member Priced Attachment Rate by month — only for annual/gift-plan members who will get credits
# Adjusted denominator: only count members whose plan is still active in that month

# 1. Members who will get credits with their end_date
df_credit_members = (
    df_tickets_per_member
    .select(
        F.col("cde_id").cast("string").alias("cde_id"),
        F.col("end_date")
    )
)

# 2. Transaction data joined with member end dates
df_txn_members = (
    spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")
    .filter((F.col("VisitDateId") >= 20210101) & (F.col("VisitDateId") <= 20260715))
    .withColumn("cde_id_str", F.col("CDE_ID").cast("string"))
    .join(df_credit_members, F.col("cde_id_str") == df_credit_members.cde_id, "inner")
    .drop(df_credit_members.cde_id)
    .withColumn("visit_date", F.to_date(F.col("VisitDateId").cast("string"), "yyyyMMdd"))
    .withColumn("txn_month", F.date_trunc("month", F.col("visit_date")))
)

# 3. CC Visits by month (only for members active in that month: end_date >= txn_month)
df_cc_visits_monthly = (
    df_txn_members
    .filter(F.col("CineClubFLAG") == "Yes")
    .filter(F.col("end_date") >= F.col("txn_month"))  # member still active
    .select("cde_id_str", "VisitDateId", "LocationId", "txn_month")
    .dropDuplicates(["cde_id_str", "VisitDateId", "LocationId"])
    .groupBy("txn_month")
    .agg(F.count("*").alias("CC_Visits"))
)

# 4. Member Priced Tickets by month (Discount Type 5, active members only)
df_member_priced_monthly = (
    df_txn_members
    .filter(F.col("CineClub_Discount_Type_ID") == 5)
    .filter(F.col("end_date") >= F.col("txn_month"))  # member still active
    .groupBy("txn_month")
    .agg(F.sum("Quantity").alias("Member_Priced_Tickets"))
)

# 5. Active members per month (for context)
df_active_members_monthly = (
    df_txn_members
    .filter(F.col("CineClubFLAG") == "Yes")
    .filter(F.col("end_date") >= F.col("txn_month"))
    .select("cde_id_str", "VisitDateId", "LocationId", "txn_month")
    .dropDuplicates(["cde_id_str", "VisitDateId", "LocationId"])
    .groupBy("txn_month")
    .agg(F.countDistinct("cde_id_str").alias("Active_Members_Visiting"))
)

# 6. Join and compute attachment rate
df_attach_rate_monthly = (
    df_cc_visits_monthly
    .join(df_member_priced_monthly, "txn_month", "left")
    .join(df_active_members_monthly, "txn_month", "left")
    .fillna(0, subset=["Member_Priced_Tickets"])
    .withColumn("Attach_Rate", 
        F.when(F.col("CC_Visits") > 0,
            F.round(F.col("Member_Priced_Tickets") / F.col("CC_Visits"), 4)
        )
    )
    .withColumn("Month_Label", F.date_format(F.col("txn_month"), "MMM yyyy"))
    .orderBy("txn_month")
    .select("Month_Label", "Active_Members_Visiting", "CC_Visits", "Member_Priced_Tickets", "Attach_Rate")
)

display(df_attach_rate_monthly)

In [0]:
# Project Member Priced Attach Rate month-by-month for credit-eligible members
# Similar to Cell 9: use per-member velocity and project until credits end
import pandas as pd
import numpy as np

# 1. Per credit-eligible member: get their monthly CC visit rate and attach rate
df_proj = (
    df_step4
    .join(
        df_tickets_per_member.select(
            F.col("cde_id").alias("cde_id_pm"),
            F.col("end_date"),
            F.col("999_Tickets_Credited")
        ),
        df_step4.cde_id == F.col("cde_id_pm"),
        "inner"
    )
    .drop("cde_id_pm")
    .withColumn("Monthly_CC_Visits", F.coalesce(F.col("P12M_CC_Visits"), F.lit(0)) / 12)
    .withColumn("Personal_Attach_Rate",
        F.when(F.col("P12M_CC_Visits") > 0,
            F.col("P12M_Member_Priced") / F.col("P12M_CC_Visits")
        ).otherwise(F.lit(0))
    )
    .select("cde_id", "end_date", "999_Tickets_Credited", 
            "Monthly_CC_Visits", "Personal_Attach_Rate")
)

# Collect to pandas
proj_members = df_proj.toPandas()
proj_members["Monthly_CC_Visits"] = proj_members["Monthly_CC_Visits"].astype(float)
proj_members["Personal_Attach_Rate"] = proj_members["Personal_Attach_Rate"].astype(float)
proj_members["end_date"] = pd.to_datetime(proj_members["end_date"])
proj_members["Credit_Months"] = proj_members["999_Tickets_Credited"].astype(int)

# 2. Month-by-month projection: for each month, only include members still active
start = pd.Timestamp("2026-07-01")
max_months = proj_members["Credit_Months"].max()

monthly_attach = []

for t in range(max_months):
    month_start = start + pd.DateOffset(months=t)
    
    # Active members: their end_date is beyond this month (still getting credits)
    active = proj_members[proj_members["end_date"] >= month_start]
    
    if len(active) == 0:
        break
    
    # Expected CC visits this month from active members
    expected_cc_visits = active["Monthly_CC_Visits"].sum()
    
    # Expected Member Priced tickets = each member's CC visits * their attach rate
    expected_member_priced = (active["Monthly_CC_Visits"] * active["Personal_Attach_Rate"]).sum()
    
    # Month-level attach rate
    attach_rate = expected_member_priced / expected_cc_visits if expected_cc_visits > 0 else 0
    
    monthly_attach.append({
        "Month": t + 1,
        "Calendar_Month": month_start.strftime("%B %Y"),
        "Active_Members": len(active),
        "Expected_CC_Visits": round(expected_cc_visits, 0),
        "Expected_Member_Priced": round(expected_member_priced, 0),
        "Projected_Attach_Rate": round(attach_rate, 4)
    })

df_attach_proj = pd.DataFrame(monthly_attach)

# 3. Display
print("=" * 80)
print("PROJECTED MEMBER PRICED ATTACH RATE BY MONTH (Credit-Eligible Members Only)")
print("=" * 80)
print(f"\n{'Month':<5} {'Calendar':<18} {'Active Mbrs':>12} {'Exp CC Visits':>14} {'Exp Mbr Priced':>15} {'Attach Rate':>12}")
print("-" * 80)

for _, row in df_attach_proj.iterrows():
    print(f"{int(row['Month']):<5} {row['Calendar_Month']:<18} {int(row['Active_Members']):>12,} {int(row['Expected_CC_Visits']):>14,} {int(row['Expected_Member_Priced']):>15,} {row['Projected_Attach_Rate']:>12.4f}")

print(f"\n--- Summary ---")
print(f"Total Expected CC Visits over credit period: {df_attach_proj['Expected_CC_Visits'].sum():,.0f}")
print(f"Total Expected Member Priced Tickets:        {df_attach_proj['Expected_Member_Priced'].sum():,.0f}")
print(f"Overall Weighted Attach Rate:                {df_attach_proj['Expected_Member_Priced'].sum() / df_attach_proj['Expected_CC_Visits'].sum():.4f}")